## Análise de Vídeo Especializada para Saúde da Mulher

Este notebook integra os modelos de vídeos treinados para a análise de vídeo especializada saúde da mulher.

Abaixo estão todos os modos disponíveis:

| Modo | Detecta |
|---|---|
| `todos` | todos os 4 modelos em sequência |
| `instrumentos` | Pinça Grasper, Gancho, Tesoura, Clipador |
| `areas-criticas` | Ovário, Mama |
| `sangramento` | sangramento em campo cirúrgico |
| `automutilacao` | Faca/Lâmina, Arma de Fogo |

In [34]:
# =====================================================================
#  INSTALAÇÃO DE DEPENDÊNCIAS
# =====================================================================
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics==8.0.149 opencv-python-headless
!pip install -q yt-dlp

In [35]:
# =====================================================================
#  IMPORTS
# =====================================================================
import importlib.util
import os
import sys
import subprocess
import relatorio as _rel
from IPython.display import Markdown, display
from datetime import datetime

In [ ]:
# =====================================================================
#  CONFIGURAÇÕES
# =====================================================================

# --- FONTE DO VÍDEO (escolha uma das opções abaixo) ---

# Opção 1: URL do YouTube (deixe vazio "" para usar vídeo local)
YOUTUBE_URL = ""

# Opção 2: Caminho para vídeo local (deixe vazio "" para baixar do YouTube)
#   Exemplos:
#     VIDEO_LOCAL = "video_test.mp4"                    # arquivo na raiz do projeto
#     VIDEO_LOCAL = r"C:\Videos\cirurgia.mp4"           # caminho absoluto
VIDEO_LOCAL = "video_tech_example.mp4"

# Nome do arquivo de trabalho quando baixado do YouTube
VIDEO_FILENAME = "video_notebook.mp4"

# Modelo a executar — escolha uma das opções abaixo:
#
#   "todos"          → todos os 4 modelos em sequência (relatório consolidado)
#   "instrumentos"   → Pinça Grasper | Gancho | Tesoura | Clipador
#   "areas-criticas" → Ovário | Mama
#   "sangramento"    → sangramento em campo cirúrgico/endoscópico
#   "automutilacao"  → Faca/Lâmina | Arma de Fogo
#
MODO = "instrumentos"

In [37]:
# =====================================================================
#  CONFIGURAÇÃO
# =====================================================================
PROJECT_ROOT = os.path.abspath(".")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(PROJECT_ROOT, ".env"))
except ImportError:
    pass

_VALID_MODES = ["todos", "instrumentos", "areas-criticas", "sangramento", "automutilacao"]
if MODO not in _VALID_MODES:
    raise ValueError(f"MODO inválido: '{MODO}'. Escolha entre: {_VALID_MODES}")

_DETECTORS_CONFIG = {
    "instrumentos":   ("src/detectors/instrumentos.py",   "InstrumentosDetector",  "instrumentos"),
    "areas-criticas": ("src/detectors/areas_criticas.py", "AreasCriticasDetector", "areas_criticas"),
    "sangramento":    ("src/detectors/sangramento.py",    "SangramentoDetector",   "sangramento"),
    "automutilacao":  ("src/detectors/automutilacao.py",  "AutomutilacaoDetector", "automutilacao"),
}

_MODES_TO_RUN = list(_DETECTORS_CONFIG.keys()) if MODO == "todos" else [MODO]

print(f"Raiz do notebook : {PROJECT_ROOT}")
print(f"Modo selecionado: {MODO}")
print(f"Modelos a rodar : {_MODES_TO_RUN}")

Raiz do notebook : f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4
Modo selecionado: sangramento
Modelos a rodar : ['sangramento']


In [ ]:
# =====================================================================
#  FONTE DO VÍDEO
# =====================================================================
_use_local = bool(VIDEO_LOCAL and VIDEO_LOCAL.strip())
_use_youtube = bool(YOUTUBE_URL and YOUTUBE_URL.strip())

if not _use_local and not _use_youtube:
    raise ValueError("Preencha VIDEO_LOCAL ou YOUTUBE_URL na célula de configuração.")
if _use_local and _use_youtube:
    print("Ambas as fontes preenchidas — usando vídeo local (VIDEO_LOCAL tem prioridade).")
    _use_youtube = False

if _use_local:
    # Resolve caminho: absoluto ou relativo à raiz do projeto
    _local_path = VIDEO_LOCAL.strip()
    if not os.path.isabs(_local_path):
        _local_path = os.path.join(PROJECT_ROOT, _local_path)
    if not os.path.exists(_local_path):
        raise FileNotFoundError(f"Vídeo local não encontrado: {_local_path}")
    VIDEO_PATH = _local_path
    size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
    print(f"Usando vídeo local: {VIDEO_PATH} ({size_mb:.1f} MB)")

else:
    VIDEO_PATH = os.path.join(PROJECT_ROOT, VIDEO_FILENAME)
    if os.path.exists(VIDEO_PATH):
        os.remove(VIDEO_PATH)
    print(f"Baixando: {YOUTUBE_URL}")
    result = subprocess.run(
        [
            "yt-dlp",
            "-f", "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4][height<=720]",
            "--merge-output-format", "mp4",
            "-o", VIDEO_PATH,
            YOUTUBE_URL,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout[-2000:])
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("Falha no download. Verifique a URL e se yt-dlp está instalado.")
    size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
    print(f"Concluído: {VIDEO_FILENAME} ({size_mb:.1f} MB)")

In [39]:
# =====================================================================
#  VERIFICAÇÃO DOS MODELOS GERADOS ANTERIORMENTE 
#  (executar app.py train --mode <modo> para gerar os modelos)
#  Recomendado GPU com suporte a CUDA
#  Os modelos já treinados aqui foram rodados localmente numa GPU NVIDIA
# =====================================================================
faltando = []
for mode in _MODES_TO_RUN:
    _, _, folder = _DETECTORS_CONFIG[mode]
    pt = os.path.join(PROJECT_ROOT, "model", folder, "weights", "best.pt")
    status = "✓" if os.path.exists(pt) else "AUSENTE"
    if status == "AUSENTE":
        faltando.append(mode)
    print(f"  [{status}] {mode}")

if faltando:
    print(f"\nAVISO: modelo(s) ausente(s): {faltando}")
    print("  Execute: python app.py train --mode <modo>")
else:
    print("\nTodos os modelos foram encontrados com sucesso.")

  [✓] sangramento

Todos os modelos foram encontrados com sucesso.


In [40]:
# =====================================================================
#  EXECUÇÃO DOS MODELOS DE DETECÇÃO
# =====================================================================
def _load_module(name, rel_path):
    path = os.path.join(PROJECT_ROOT, rel_path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

model_results = []
single_result = None

total = len(_MODES_TO_RUN)
for i, mode in enumerate(_MODES_TO_RUN, 1):
    print(f"\n{'='*60}")
    print(f"  [{i}/{total}] {mode}")
    print(f"{'='*60}")

    rel_path, cls_name, folder = _DETECTORS_CONFIG[mode]
    mod = _load_module(mode, rel_path)
    detector = getattr(mod, cls_name)()
    result = detector.detect_video(VIDEO_PATH, headless=True)

    if result is None:
        print(f"  AVISO: {mode} pulado (modelo ausente ou erro).")
        continue

    pct = len(result["anomalies"]) / max(result["frame_count"], 1) * 100
    print(f"  Frames: {result['frame_count']}  |  "
          f"Detecções: {result['detections_count']}  |  "
          f"Anomalias: {len(result['anomalies'])} ({pct:.1f}%)")

    entry = {
        "model_folder":  folder,
        "frame_count":   result["frame_count"],
        "detections":    result["detections_count"],
        "anomalies":     result["anomalies"],
        "fps":           result["fps"],
        "class_summary": result.get("class_summary", {}),
    }
    model_results.append(entry)
    if MODO != "todos":
        single_result = result

print(f"\n{'='*60}")
print(f"  Concluído: {len(model_results)}/{total} modelo(s) processado(s)")
print(f"{'='*60}")


  [1/1] sangramento
Usando modelo: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\model\sangramento\weights\best.pt
Iniciando análise [bleeding_detector] ...
Vídeo: video_notebook.mp4 | 854x480 @ 24.0fps
Relatório TXT salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.txt
Relatório HTML salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.html
Relatório JSON salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.json

=== RESULTADO FINAL [bleeding_detector] ===
Frames analisados: 1326
Detecções totais:  843
Média por frame:   0.64
Anomalias:         11
Taxa de anomalia:  0.83%

Arquivos gerados em: saida/sangramento/
  Vídeo anotado: output.mp4
  Relatório:     relatorio.txt/.html/.json
  Frames: 1326  |  Detecções: 843  |  Anomalias: 11 (0.8%)

  Concluído: 1/1 modelo(s) processado(s)


In [41]:
# =====================================================================
#  GERAÇÃO E EXIBIÇÃO DO RELATÓRIO
# =====================================================================
import relatorio as _rel
from IPython.display import Markdown, display
from datetime import datetime

def _ts(frame, fps):
    total = int(frame) // max(int(fps), 1)
    return f"{total // 60:02d}:{total % 60:02d}"

def _render_single(result, mode, fps, video_path):
    frames    = result["frame_count"]
    dets      = result["detections_count"]
    anomalies = result["anomalies"]
    avg       = dets / frames if frames else 0
    rate      = len(anomalies) / frames * 100 if frames else 0
    duration  = _ts(frames, fps)
    risk, _   = _rel._get_risk_level(rate)
    now       = datetime.now().strftime("%d/%m/%Y %H:%M")
    vname     = os.path.basename(video_path) if video_path else "N/A"

    critico = _rel._count_by_severity(anomalies, "CRÍTICO")
    alto    = _rel._count_by_severity(anomalies, "ALTO")
    medio   = _rel._count_by_severity(anomalies, "MÉDIO")

    md = []
    md.append(f"# Relatório de Análise — {mode.replace('-', ' ').title()}")
    md.append(f"**Vídeo:** `{vname}` | **Duração:** {duration} | **FPS:** {fps:.1f} | **Data:** {now}")
    md.append("")

    md.append("## Resumo Estatístico")
    md.append("| Métrica | Valor |")
    md.append("|---|---|")
    md.append(f"| Frames analisados | {frames} |")
    md.append(f"| Detecções totais | {dets} |")
    md.append(f"| Média por frame | {avg:.2f} |")
    md.append(f"| Anomalias detectadas | {len(anomalies)} |")
    md.append(f"| Taxa de anomalia | {rate:.1f}% |")
    md.append(f"| Nível de risco | **{risk}** |")
    md.append("")

    md.append("## Severidade das Anomalias")
    md.append("| Crítico | Alto | Médio |")
    md.append("|:---:|:---:|:---:|")
    md.append(f"| {critico} | {alto} | {medio} |")
    md.append("")

    cs = result.get("class_summary", {})
    if cs:
        md.append("## Objetos Detectados")
        md.append("| Classe | Ocorrências | Primeiro | Último | % Frames |")
        md.append("|---|---:|:---:|:---:|---:|")
        for name, info in sorted(cs.items(), key=lambda x: -x[1]["count"]):
            md.append(f"| {name} | {info['count']} | {_ts(info['first_frame'], fps)} | {_ts(info['last_frame'], fps)} | {info['frames_pct']:.1f}% |")
        md.append("")

    md.append("## Linha do Tempo de Anomalias")
    if not anomalies:
        md.append("_Nenhuma anomalia detectada durante o procedimento._")
    else:
        md.append("| Timestamp | Frame | Severidade | Tipo | Descrição |")
        md.append("|:---:|:---:|:---:|:---:|---|")
        for a in anomalies:
            if isinstance(a, dict):
                f = a.get("frame", 0)
                sev = a.get("severity", "-")
                md.append(f"| {_ts(f, fps)} | {f} | {sev} | {a.get('type','-')} | {a.get('description','')} |")
    md.append("")

    md.append("## Avaliação Clínica")
    absences   = _rel._count_by_type(anomalies, "AUSÊNCIA")
    excesses   = _rel._count_by_type(anomalies, "EXCESSO")
    variations = _rel._count_by_type(anomalies, "VARIAÇÃO")
    md.append("| Tipo | Contagem |")
    md.append("|---|:---:|")
    md.append(f"| Ausências prolongadas | {absences} |")
    md.append(f"| Excessos detectados | {excesses} |")
    md.append(f"| Variações bruscas | {variations} |")
    md.append("")

    md.append("## Recomendação")
    if rate > 20:
        md.append("> **ATENCAO CRITICA:** Alta taxa de anomalias. Revisão imediata necessária.")
    elif rate > 10:
        md.append("> **ATENCAO:** Taxa elevada. Revisão por especialista recomendada.")
    elif rate > 5:
        md.append("> **OBSERVACAO:** Anomalias dentro de limite aceitável, mas merecem atenção.")
    else:
        md.append("> Procedimento dentro dos parâmetros normais. Nenhuma ação recomendada.")
    if absences: md.append(f"> - Revisar **{absences}** segmento(s) com ausência prolongada.")
    if excesses: md.append(f"> - Verificar **{excesses}** frame(s) com excesso detectado.")
    if variations: md.append(f"> - Investigar **{variations}** variação(ões) brusca(s).")

    return "\n".join(md)


def _render_combined(model_results, video_path, fps):
    vname  = os.path.basename(video_path) if video_path else "N/A"
    now    = datetime.now().strftime("%d/%m/%Y %H:%M")
    frames = model_results[0]["frame_count"] if model_results else 0
    total_anomalies = sum(len(r["anomalies"]) for r in model_results)
    rate   = total_anomalies / frames * 100 if frames else 0
    risk, _ = _rel._get_risk_level(rate)
    duration = _ts(frames, fps)

    md = []
    md.append("# Relatório Consolidado — Análise Cirúrgica Completa")
    md.append(f"**Vídeo:** `{vname}` | **Duração:** {duration} | **Frames:** {frames} | **Data:** {now}")
    md.append("")

    md.append("## Visão Geral")
    md.append("| Métrica | Valor |")
    md.append("|---|---|")
    md.append(f"| Frames analisados | {frames} |")
    md.append(f"| Modelos executados | {len(model_results)} |")
    md.append(f"| Anomalias totais | {total_anomalies} |")
    md.append(f"| Taxa geral | {rate:.1f}% |")
    md.append(f"| Risco geral | **{risk}** |")
    md.append("")

    md.append("## Critérios Especializados")
    for ev in _rel._evaluate_criteria(model_results):
        c = ev["criterion"]
        if ev["triggered"]:
            md.append(f"### [ACIONADO] {c['title']}")
            md.append(f"_{c['description']}_")
            md.append("")
            md.append("| Modelo | Ocorrências | Crítico | Alto | Médio |")
            md.append("|---|:---:|:---:|:---:|:---:|")
            for fd in ev["findings"]:
                md.append(f"| {fd['folder']} | {fd['count']} | {fd['critico']} | {fd['alto']} | {fd['medio']} |")
            md.append("")
            md.append(f"> **Recomendação:** {c['recommendation']}")
        else:
            md.append(f"### {c['title']} — Sem ocorrências")
            md.append(f"_{c['description']}_")
        md.append("")

    md.append("## Resultados por Modelo")
    md.append("| Modelo | Detecções | Anomalias | Taxa | Risco |")
    md.append("|---|:---:|:---:|:---:|:---:|")
    for r in model_results:
        arate = len(r["anomalies"]) / r["frame_count"] * 100 if r["frame_count"] else 0
        mlabel, _ = _rel._get_risk_level(arate)
        md.append(f"| {r['model_folder']} | {r['detections']} | {len(r['anomalies'])} | {arate:.1f}% | {mlabel} |")
    md.append("")

    for r in model_results:
        fps_r = r.get("fps", fps)
        cs = r.get("class_summary", {})
        if cs:
            md.append(f"### Objetos — {r['model_folder']}")
            md.append("| Classe | Ocorrências | Primeiro | Último | % Frames |")
            md.append("|---|:---:|:---:|:---:|:---:|")
            for name, info in sorted(cs.items(), key=lambda x: -x[1]["count"]):
                md.append(f"| {name} | {info['count']} | {_ts(info['first_frame'], fps_r)} | {_ts(info['last_frame'], fps_r)} | {info['frames_pct']:.1f}% |")
            md.append("")

    all_anomalies = []
    for r in model_results:
        fps_r = r.get("fps", fps)
        for a in r["anomalies"]:
            if isinstance(a, dict):
                all_anomalies.append((r["model_folder"], a, fps_r))

    md.append("## Linha do Tempo — Todas as Anomalias")
    if not all_anomalies:
        md.append("_Nenhuma anomalia detectada em nenhum modelo._")
    else:
        md.append("| Modelo | Timestamp | Frame | Severidade | Tipo | Descrição |")
        md.append("|:---:|:---:|:---:|:---:|:---:|---|")
        for folder, a, fps_r in sorted(all_anomalies, key=lambda x: x[1].get("frame", 0)):
            f = a.get("frame", 0)
            sev = a.get("severity", "-")
            md.append(f"| {folder} | {_ts(f, fps_r)} | {f} | {sev} | {a.get('type','-')} | {a.get('description','')} |")

    return "\n".join(md)


# --- Executar ---
if not model_results:
    display(Markdown("### Nenhum resultado disponível para gerar relatório."))
else:
    saida_dir = os.path.join(PROJECT_ROOT, "saida")
    os.makedirs(saida_dir, exist_ok=True)
    fps = model_results[0].get("fps", 20) if model_results else 20

    if MODO == "todos":
        report_txt = os.path.join(saida_dir, "relatorio_geral.txt")
        _rel.generate_combined_report(report_txt, model_results, VIDEO_PATH)
        md_output = _render_combined(model_results, VIDEO_PATH, fps)
    else:
        _, _, folder = _DETECTORS_CONFIG[MODO]
        model_saida = os.path.join(saida_dir, folder)
        os.makedirs(model_saida, exist_ok=True)
        report_txt = os.path.join(model_saida, "relatorio.txt")
        _rel.generate_report(
            report_txt,
            single_result["frame_count"],
            single_result["detections_count"],
            single_result["anomalies"],
            fps=single_result["fps"],
            video_path=VIDEO_PATH,
            class_summary=single_result.get("class_summary", {}),
        )
        fps = single_result.get("fps", 20)
        md_output = _render_single(single_result, MODO, fps, VIDEO_PATH)

    display(Markdown(md_output))

Relatório TXT salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.txt
Relatório HTML salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.html
Relatório JSON salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.json


# Relatório de Análise — Sangramento
**Vídeo:** `video_notebook.mp4` | **Duração:** 00:57 | **FPS:** 24.0 | **Data:** 12/05/2026 18:42

## Resumo Estatístico
| Métrica | Valor |
|---|---|
| Frames analisados | 1326 |
| Detecções totais | 843 |
| Média por frame | 0.64 |
| Anomalias detectadas | 11 |
| Taxa de anomalia | 0.8% |
| Nível de risco | **BAIXO** |

## Severidade das Anomalias
| Crítico | Alto | Médio |
|:---:|:---:|:---:|
| 0 | 3 | 8 |

## Objetos Detectados
| Classe | Ocorrências | Primeiro | Último | % Frames |
|---|---:|:---:|:---:|---:|
| Sangramento | 843 | 00:01 | 00:55 | 63.6% |

## Linha do Tempo de Anomalias
| Timestamp | Frame | Severidade | Tipo | Descrição |
|:---:|:---:|:---:|:---:|---|
| 00:01 | 31 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:14 | 340 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:14 | 344 | ALTO | SANGRAMENTO | Sangramento contínuo detectado (5 frames consecutivos) — monitorar evolução; possível complicação em procedimento ginecológico/obstétrico |
| 00:31 | 730 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:31 | 733 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:31 | 735 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:32 | 739 | ALTO | SANGRAMENTO | Sangramento contínuo detectado (5 frames consecutivos) — monitorar evolução; possível complicação em procedimento ginecológico/obstétrico |
| 00:32 | 741 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:32 | 745 | ALTO | SANGRAMENTO | Sangramento contínuo detectado (5 frames consecutivos) — monitorar evolução; possível complicação em procedimento ginecológico/obstétrico |
| 00:54 | 1243 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |
| 00:55 | 1270 | MÉDIO | SANGRAMENTO | Sangramento detectado no campo — aguardando confirmação de desvio em procedimento cirúrgico/obstétrico |

## Avaliação Clínica
| Tipo | Contagem |
|---|:---:|
| Ausências prolongadas | 0 |
| Excessos detectados | 0 |
| Variações bruscas | 0 |

## Recomendação
> Procedimento dentro dos parâmetros normais. Nenhuma ação recomendada.